# Suite2p — batch processing

Load one settings file, then run the same settings on several recordings.

* `settings`: **how** to process (shared by all recordings).
* `db`: **what** to process (one per recording: data and save paths, planes, channels).

Before running: 
* Install Jupyter and Python VSCode extensions
* remember to set the conda/suite2p env as the python kernel.

In [1]:
import json
import logging
import shutil
from pathlib import Path
import suite2p
from suite2p import default_db
from suite2p.run_s2p import logger_setup, get_save_folder

## 1. Load the settings

In [2]:
settings_path = r"C:\Users\User\Documents\Code\suite2p\Python_demo_Federico\Dendrites_iGluSNFR4s_settings.json"

with open(settings_path) as f:
    settings = json.load(f)

## 2. One `db` per recording

In [ ]:
dbs = [
    {
        "data_path":  [
            r"Z:\Lab_Share\suite2p_tutorial\Data\2P_invivo\AL042\20260606\1",
        ],
            "fast_disk":  "C:\\",   # local disk for the temporary binary (a raw string can't end in a backslash)
        "save_path0": r"Z:\Lab_Share\Federico\suite2p_samples\processed\AL042\20260606",
    },
 
    {
        "data_path":  [r"Z:\Lab_Share\suite2p_tutorial\Data\2P_invivo\NM035\20260513\1"],
            "fast_disk":  "C:\\",   # local disk for the temporary binary (a raw string can't end in a backslash)
        "save_path0": r"Z:\Lab_Share\Federico\suite2p_samples\processed\NM035\20260513",
    },
]

## 3. Run

If a recording already has results, suite2p reuses the old `reg_outputs.npy` and **skips registration**, even when it has just rebuilt an unregistered binary from the TIFFs. To avoid processing an unaligned movie:

* `OVERWRITE = False`: recordings that already have results are skipped.
* `OVERWRITE = True`: that recording's `suite2p` folder is deleted and processed from scratch.

In [6]:
OVERWRITE = True

s2p_logger = logging.getLogger("suite2p")

for db in dbs:
    db = {**default_db(), **db}
    save_folder = Path(get_save_folder(db))

    if any(save_folder.glob("plane*/reg_outputs.npy")):
        if not OVERWRITE:
            print(f"SKIPPING (already processed, set OVERWRITE = True to redo): {save_folder}")
            continue
        print(f"Deleting previous results: {save_folder}")
        shutil.rmtree(save_folder)

    # run_s2p() doesn't set up logging itself (the CLI/GUI do), so without this
    # the per-stage messages are not shown. Drop the previous recording's log
    # file handler so each run.log only contains its own recording.
    for h in [h for h in s2p_logger.handlers if isinstance(h, logging.FileHandler)]:
        s2p_logger.removeHandler(h)
        h.close()
    logger_setup(str(save_folder))   # prints to this cell + writes <save_folder>/run.log

    suite2p.run_s2p(settings=settings, db=db)

2026-11-12 15:18:49,271 [INFO] 
cellpose version: 	1.0.0.1 
platform:       	win32 
python version: 	3.11.14 
torch version:  	2.10.0+cu126
2026-11-12 15:18:49,272 [INFO] data_path: ['Z:\\Lab_Share\\suite2p_tutorial\\Data\\2P_invivo\\AL042\\20260606\\1']
2026-11-12 15:18:49,275 [INFO] ** Found 1 files - converting to binary **


creating log file Z:\Lab_Share\Federico\suite2p_samples\processed\AL042\20260606\suite2p\run.log


2026-11-12 15:19:24,849 [INFO] 2000 frames of binary, time 35.52 sec.
2026-11-12 15:19:25,040 [INFO] Wrote 2000 frames per binary, 1 folders + 1 channels, 35.77sec
2026-11-12 15:19:25,088 [INFO] >>>>>>>>>>>>>>>>>>>>> PLANE 0 <<<<<<<<<<<<<<<<<<<<<<
2026-11-12 15:19:25,260 [INFO] ** torch.device('cuda') installed and working. **
2026-11-12 15:19:25,263 [INFO] Running registration
2026-11-12 15:19:25,267 [INFO] binary output path: C:\suite2p\plane0\data.bin
2026-11-12 15:19:25,278 [INFO] NOTE: applying default classifier: C:\Users\User\.suite2p\classifiers\classifier_user.npy
2026-11-12 15:19:25,280 [INFO] ----------- REGISTRATION
2026-11-12 15:19:25,280 [INFO] registering 1 channels
2026-11-12 15:19:28,098 [INFO] Reference frame, 2.70 sec.
2026-11-12 15:19:28,125 [INFO] Registering 2000 frames in 20 batches
2026-11-12 15:19:32,059 [INFO] 100%|##########| 20/20 [00:03<00:00,  5.09it/s]
2026-11-12 15:19:32,117 [INFO] ----------- Total 6.84 sec
2026-11-12 15:19:42,143 [INFO] Registration me

creating log file Z:\Lab_Share\Federico\suite2p_samples\processed\20260513\NM035\suite2p\run.log


2026-11-12 15:21:00,211 [INFO] 2000 frames of binary, time 20.46 sec.
2026-11-12 15:21:00,411 [INFO] Wrote 2000 frames per binary, 1 folders + 1 channels, 20.69sec
2026-11-12 15:21:00,467 [INFO] >>>>>>>>>>>>>>>>>>>>> PLANE 0 <<<<<<<<<<<<<<<<<<<<<<
2026-11-12 15:21:00,469 [INFO] ** torch.device('cuda') installed and working. **
2026-11-12 15:21:00,472 [INFO] Running registration
2026-11-12 15:21:00,474 [INFO] binary output path: C:\suite2p\plane0\data.bin
2026-11-12 15:21:00,479 [INFO] NOTE: applying default classifier: C:\Users\User\.suite2p\classifiers\classifier_user.npy
2026-11-12 15:21:00,480 [INFO] ----------- REGISTRATION
2026-11-12 15:21:00,480 [INFO] registering 1 channels
2026-11-12 15:21:03,063 [INFO] Reference frame, 2.47 sec.
2026-11-12 15:21:03,094 [INFO] Registering 2000 frames in 20 batches
2026-11-12 15:21:06,715 [INFO] 100%|##########| 20/20 [00:03<00:00,  5.53it/s]
2026-11-12 15:21:06,773 [INFO] ----------- Total 6.29 sec
2026-11-12 15:21:16,660 [INFO] Registration me